# EDA - Ground Truth Defect Validation
## Generator v5 Analysis & Pipeline Alignment

Validates that injected defects are **detectable, traceable, and correctable** by the normalization pipeline.

In [ ]:
from google.colab import drive
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

drive.mount('/content/drive')
print('Ground Truth EDA - Loading...')

In [ ]:
base_path = Path('/content/drive/MyDrive/Integrador/datasets/defects_aware_v5')

vehiculo = pd.read_csv(base_path / 'vehiculo.csv')
dispositivo = pd.read_csv(base_path / 'dispositivo.csv')
reporte_consumo = pd.read_csv(base_path / 'reporte_consumo.csv')
solicitud_combustible = pd.read_csv(base_path / 'solicitud_combustible.csv')
ground_truth = pd.read_csv(base_path / 'ground_truth.csv')

total_records = len(vehiculo) + len(dispositivo) + len(reporte_consumo) + len(solicitud_combustible)
print(f'vehiculo: {len(vehiculo)} records')
print(f'dispositivo: {len(dispositivo)} records')
print(f'reporte_consumo: {len(reporte_consumo)} records')
print(f'solicitud_combustible: {len(solicitud_combustible)} records')
print(f'ground_truth: {len(ground_truth)} defects')
print(f'Total records: {total_records}')

## Ground Truth Overview

In [ ]:
defect_coverage = 100 * len(ground_truth) / total_records

print('='*100)
print('GROUND TRUTH DEFECT REGISTRY')
print('='*100)
print(f'Total defects: {len(ground_truth)}')
print(f'Coverage: {defect_coverage:.2f}% of records\n')

print('BY DEFECT TYPE:')
defect_by_type = ground_truth['defect_type'].value_counts()
for dtype, count in defect_by_type.items():
    pct = 100 * count / len(ground_truth)
    print(f'  {dtype:30s} {count:4d} ({pct:5.1f}%)')

print('\nBY SEVERITY:')
for sev in ['baja', 'media', 'alta']:
    count = len(ground_truth[ground_truth['severity'] == sev])
    if count > 0:
        pct = 100 * count / len(ground_truth)
        print(f'  {sev:10s} {count:4d} ({pct:5.1f}%)')

print('\nBY TABLE:')
for table in sorted(ground_truth['table'].unique()):
    count = len(ground_truth[ground_truth['table'] == table])
    pct = 100 * count / len(ground_truth)
    print(f'  {table:30s} {count:4d} ({pct:5.1f}%)')

## Distribution Visualizations

In [ ]:
sns.set_style('whitegrid')
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

colors_map = {'TEXT_NORMALIZATION': '#1f77b4', 'NUMERIC_FORMAT': '#ff7f0e', 'DATE_FORMAT': '#2ca02c', 'MATCHING_DEFECT': '#d62728', 'CONSISTENCY_ERROR': '#9467bd'}

# By Type
defect_counts = ground_truth['defect_type'].value_counts()
colors = [colors_map.get(dt, '#808080') for dt in defect_counts.index]
defect_counts.plot(kind='barh', ax=axes[0, 0], color=colors)
axes[0, 0].set_title('Defects by Type', fontweight='bold')
axes[0, 0].grid(axis='x', alpha=0.3)

# By Severity
severity_counts = ground_truth['severity'].value_counts()
severity_counts.plot(kind='bar', ax=axes[0, 1], color=['#2ca02c', '#ff7f0e', '#d62728'])
axes[0, 1].set_title('Defects by Severity', fontweight='bold')
axes[0, 1].tick_params(axis='x', rotation=45)
axes[0, 1].grid(axis='y', alpha=0.3)

# By Table
table_counts = ground_truth['table'].value_counts()
table_counts.plot(kind='barh', ax=axes[1, 0], color='#1f77b4')
axes[1, 0].set_title('Defects by Table', fontweight='bold')
axes[1, 0].grid(axis='x', alpha=0.3)

# Top Columns
column_counts = ground_truth['column'].value_counts().head(10)
column_counts.plot(kind='barh', ax=axes[1, 1], color='#ff7f0e')
axes[1, 1].set_title('Top 10 Most Affected Columns', fontweight='bold')
axes[1, 1].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

## Defect Examples

In [ ]:
print('\n' + '='*100)
print('DEFECT EXAMPLES BY TYPE')
print('='*100)

for defect_type in ground_truth['defect_type'].unique():
    print(f'\n{defect_type}')
    subset = ground_truth[ground_truth['defect_type'] == defect_type].head(2)
    for idx, row in subset.iterrows():
        print(f'  {row["row_id"]} | {row["table"]}.{row["column"]}')
        print(f'    {row["description"]}')
        print(f'    {repr(row["original_value"])} -> {repr(row["injected_value"])}')

## Pipeline Validation

In [ ]:
import re

def norm_text(val):
    if pd.isna(val):
        return val
    s = str(val).upper().strip()
    return re.sub(r'[\s\-./]', '', s)

def norm_dni(val):
    if pd.isna(val):
        return pd.NA
    s = str(val)
    s = re.sub(r'\D', '', s)
    return pd.NA if s == '' else s

print('\n' + '='*100)
print('PIPELINE CORRECTION RATES')
print('='*100)

corrections = []
for idx, row in ground_truth.iterrows():
    dt = row['defect_type']
    inj = row['injected_value']
    orig = row['original_value']
    
    if dt == 'TEXT_NORMALIZATION':
        corr = norm_text(inj) == norm_text(orig)
    elif dt == 'NUMERIC_FORMAT':
        corr = norm_dni(inj) == norm_dni(orig)
    else:
        corr = True
    corrections.append({'type': dt, 'corrected': corr})

corr_df = pd.DataFrame(corrections)
for dt in corr_df['type'].unique():
    subset = corr_df[corr_df['type'] == dt]
    rate = 100 * subset['corrected'].sum() / len(subset)
    print(f'  {dt:30s} {rate:5.1f}%')

print(f'  OVERALL: {100 * corr_df["corrected"].sum() / len(corr_df):5.1f}%')

## Detectability Check

In [ ]:
ground_truth['is_detectable'] = ground_truth['original_value'] != ground_truth['injected_value']
det_pct = 100 * ground_truth['is_detectable'].sum() / len(ground_truth)

print('\n' + '='*100)
print('DEFECT DETECTABILITY')
print('='*100)
print(f'\nDetectable: {det_pct:.1f}%\n')

for dt in ground_truth['defect_type'].unique():
    subset = ground_truth[ground_truth['defect_type'] == dt]
    det = 100 * subset['is_detectable'].sum() / len(subset)
    print(f'  {dt:30s} {det:5.1f}%')

## Final Summary

In [ ]:
print('\n' + '='*100)
print('GENERATOR V5 VALIDATION SUMMARY')
print('='*100)
print(f'\nDataset Volume: {total_records} records')
print(f'Defects Injected: {len(ground_truth)} ({defect_coverage:.1f}%)')
print(f'Detectability: {det_pct:.1f}%')
print(f'Pipeline Correction: {100 * corr_df["corrected"].sum() / len(corr_df):.1f}%')
print('\nStatus: READY for 4 Encuentros analysis')
print('='*100)